In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
from peft import get_peft_model, LoraConfig, TaskType

/Users/shreevijay/llm_hp/hp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-instruct")
model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M-instruct", device_map='auto')

In [3]:
dataset = load_dataset('Amod/mental_health_counseling_conversations')

In [4]:
model.device

device(type='mps', index=0)

In [5]:
instruction = "You are a helpful assistant. Be polite and respectful while answering questions."

def tokenize_text(ds):
    ds['text'] = tokenizer.apply_chat_template([
        {'role': 'system', 'content': instruction.strip()},
        {'role': 'user', 'content':ds['Context'].strip()}, 
        {'role': 'assistant', 'content':ds['Response'].strip()}], tokenize=False)
    return ds

In [6]:
tokenized_dataset = dataset.map(tokenize_text)
tokenized_dataset = tokenized_dataset.remove_columns(['Context', 'Response'])
tokenized_dataset = tokenized_dataset['train'].train_test_split(test_size=0.2, shuffle=True)

In [7]:
tokenized_dataset['train']['text'][0]

"<|im_start|>system\nYou are a helpful assistant. Be polite and respectful while answering questions.<|im_end|>\n<|im_start|>user\nHow does a person start the counseling process?<|im_end|>\n<|im_start|>assistant\nPhone or email a counselor whose profile you've read and which feels right for you.Ask to get a feel as to the way the person would handle your problem and work with you.In my practice I offer a phone consult which generally continues for twenty minutes.I feel it is only fair that a prospective patient has a feel for the service they are about to purchase before they can be expected to pay money for a service which may not be to their liking at all.<|im_end|>\n"

In [8]:
import torch.nn as nn

def find_all_linear_names(model):
    cls = nn.Linear
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)

modules = find_all_linear_names(model)

In [9]:
modules

['gate_proj', 'v_proj', 'down_proj', 'q_proj', 'up_proj', 'k_proj', 'o_proj']

In [11]:

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules
)
# model, tokenizer = setup_chat_format(model, tokenizer)
model = get_peft_model(model, peft_config)

/Users/shreevijay/llm_hp/hp_env/lib/python3.12/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


'NoneType' object has no attribute 'cadam32bit_grad_fp32'


In [21]:
training_arguments = TrainingArguments(
    output_dir='lora-sft-1',
    per_device_train_batch_size=2,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    # optim="paged_adamw_32bit",
    num_train_epochs=5,
    eval_strategy="epoch",
    # eval_steps=0.2,
    logging_steps=50,
    warmup_steps=10,
    logging_strategy="epoch",
    learning_rate=2e-4,
    # fp16=False,
    # bf16=False,
)

In [22]:
# Setting sft parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    peft_config=peft_config,
    # max_seq_length= 512,
    # dataset_text_field="text",
    # tokenizer=tokenizer,
    args=training_arguments,
    # packing= False,
)

In [ ]:
import torch
torch.mps.empty_cache()

trainer.train()

In [ ]:
messages = [{"role": "system", "content": instruction},
    {"role": "user", "content": "I'm very depressed. How do I find someone to talk to?"}]

prompt = tokenizer.apply_chat_template(messages, tokenize=False)
    
inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True).to('mps')

outputs = model.generate(**inputs, max_new_tokens=150, num_return_sequences=1)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(text)

system
You are a helpful assistant. Be polite and respectful while answering questions.
user
I'm very depressed. How do I find someone to talk to?
assistant
I'm sorry you are depressed.  I would suggest that you talk to someone about your depression.  If you are unable to talk to someone, then you can talk to a local mental health professional.  I would also suggest that you find a therapist who specializes in depression.  Depression is a very common condition and it can be very difficult to work through.  I would also suggest that you find a therapist who is licensed in your state.  It is important that you find a therapist who is licensed to help you work through your depression.  I would also suggest that you find a therapist who is a licensed therapist in your state.  It is important that you find a therapist
